# Final Model Comparison

This notebook evaluates and visualizes test-set performance for:

- MPNet SentenceTransformer with the original/unretrained encoder
- MPNet SentenceTransformer with the retrained encoder
- TF-IDF + One-vs-Rest Logistic Regression

It uses the same fixed split from `outputs/SBERT/data_split.npz`, the same metrics used in prior notebooks, and shows samples where MPNet is correct but TF-IDF is wrong and vice versa.


## 1. Setup


In [ ]:
from pathlib import Path
import json
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import seaborn as sns
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    hamming_loss,
    multilabel_confusion_matrix,
    precision_score,
    recall_score,
)
from sentence_transformers import SentenceTransformer

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 80)


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (
            (candidate / "data" / "Restaurant_ABSA_processed.csv").exists()
            and (candidate / "outputs" / "SBERT" / "data_split.npz").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "Restaurant_ABSA_processed.csv"
SPLIT_PATH = PROJECT_ROOT / "outputs" / "MPNet" / "data_split.npz"

MPNet_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "MPNet"
MPNet_MODEL_DIR = PROJECT_ROOT / "models" / "mpnet"
MPNet_CLASSIFIER_DIR = MPNet_MODEL_DIR / "classifiers"

TFIDF_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "TF-IDF"
TFIDF_MODEL_DIR = PROJECT_ROOT / "models" / "TF-IDF"
TFIDF_CLASSIFIER_DIR = TFIDF_MODEL_DIR / "classifiers"

FINAL_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "final_comparison"
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ASPECT_COLUMNS = ["food", "price", "service", "ambiance", "miscellaneous"]
TFIDF_ASPECT_COLUMNS = ["food", "service", "price", "ambiance", "miscellaneous"]

required_paths = [
    DATA_PATH,
    SPLIT_PATH,
    MPNet_CLASSIFIER_DIR / "unretrained" / "model.joblib",
    MPNet_CLASSIFIER_DIR / "unretrained" / "metadata.json",
    MPNet_CLASSIFIER_DIR / "retrained" / "model.joblib",
    MPNet_CLASSIFIER_DIR / "retrained" / "metadata.json",
    TFIDF_OUTPUT_DIR / "X_test_tfidf.npz",
    TFIDF_CLASSIFIER_DIR / "tfidf_ovr_logreg.joblib",
    TFIDF_CLASSIFIER_DIR / "tfidf_metadata.json",
]

missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required artifacts:\n" + "\n".join(map(str, missing_paths)))

print("Project root:", PROJECT_ROOT)
print("Final outputs:", FINAL_OUTPUT_DIR)


## 2. Load Test Set


In [ ]:
df = pd.read_csv(DATA_PATH).dropna(subset=["review_en"]).reset_index(drop=True)
split = np.load(SPLIT_PATH)
test_idx = split["test_idx"]

test_df = df.iloc[test_idx].copy().reset_index().rename(columns={"index": "source_row"})
test_texts = test_df["review_en"].astype(str).tolist()
y_test = test_df[ASPECT_COLUMNS].to_numpy(dtype=np.int8)

print(f"Test samples: {len(test_df)}")
display(test_df[["source_row", "review_en", *ASPECT_COLUMNS]].head())


## 3. Shared Evaluation Helpers


In [ ]:
def labels_to_text(row, aspect_cols=ASPECT_COLUMNS) -> str:
    row = np.asarray(row).astype(bool)
    labels = np.array(aspect_cols)[row]
    return ", ".join(labels) if len(labels) else "None"


def apply_thresholds(probabilities: np.ndarray, thresholds: dict, aspect_cols: list[str]) -> np.ndarray:
    predictions = np.zeros_like(probabilities, dtype=np.int8)
    for index, aspect in enumerate(aspect_cols):
        predictions[:, index] = (probabilities[:, index] >= thresholds[aspect]).astype(np.int8)

    empty_rows = np.flatnonzero(predictions.sum(axis=1) == 0)
    for row in empty_rows:
        predictions[row, np.argmax(probabilities[row])] = 1
    return predictions


def reorder_columns(matrix: np.ndarray, from_cols: list[str], to_cols: list[str]) -> np.ndarray:
    positions = [from_cols.index(col) for col in to_cols]
    return matrix[:, positions]


def evaluate_predictions(model_name: str, y_true: np.ndarray, y_pred: np.ndarray, seconds: float | None = None) -> dict:
    per_aspect_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)
    metrics = {
        "model": model_name,
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "samples_f1": f1_score(y_true, y_pred, average="samples", zero_division=0),
        "precision_micro": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "recall_micro": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "exact_match_ratio": accuracy_score(y_true, y_pred),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "mean_predicted_labels": y_pred.sum(axis=1).mean(),
        "mean_true_labels": y_true.sum(axis=1).mean(),
    }
    if seconds is not None:
        metrics["seconds"] = seconds
        metrics["milliseconds_per_review"] = seconds * 1000 / len(y_true)

    for index, aspect in enumerate(ASPECT_COLUMNS):
        metrics[f"{aspect}_f1"] = per_aspect_f1[index]
    return metrics


def build_prediction_frame(model_name: str, probabilities: np.ndarray, predictions: np.ndarray) -> pd.DataFrame:
    output = test_df[["source_row", "review_en", "review_cleaned"]].copy()
    output["model"] = model_name
    output["true_aspects"] = [labels_to_text(row) for row in y_test]
    output["predicted_aspects"] = [labels_to_text(row) for row in predictions]
    output["exact_match"] = (predictions == y_test).all(axis=1)
    output["true_label_count"] = y_test.sum(axis=1)
    output["predicted_label_count"] = predictions.sum(axis=1)

    for index, aspect in enumerate(ASPECT_COLUMNS):
        output[f"true_{aspect}"] = y_test[:, index]
        output[f"pred_{aspect}"] = predictions[:, index]
        output[f"prob_{aspect}"] = probabilities[:, index]

    output["max_probability"] = probabilities.max(axis=1)
    output["mean_true_aspect_probability"] = [
        probabilities[row_index, y_test[row_index].astype(bool)].mean()
        if y_test[row_index].any()
        else np.nan
        for row_index in range(len(y_test))
    ]
    return output


## 4. Evaluate MPNet Models


In [ ]:
def evaluate_mpnet_bundle(bundle_name: str) -> dict:
    bundle_dir = MPNet_CLASSIFIER_DIR / bundle_name
    classifier = joblib.load(bundle_dir / "model.joblib")
    with (bundle_dir / "metadata.json").open(encoding="utf-8") as file:
        metadata = json.load(file)

    aspect_cols = metadata["aspect_cols"]
    encoder_path = PROJECT_ROOT / metadata["encoder_path"]
    encoder = SentenceTransformer(str(encoder_path))

    start = time.perf_counter()
    embeddings = encoder.encode(
        test_texts,
        batch_size=metadata.get("batch_size", 32),
        normalize_embeddings=metadata.get("normalize_embeddings", True),
        convert_to_numpy=True,
        show_progress_bar=True,
    )
    probabilities = classifier.predict_proba(embeddings)
    seconds = time.perf_counter() - start

    predictions = apply_thresholds(probabilities, metadata["thresholds"], aspect_cols)
    probabilities = reorder_columns(probabilities, aspect_cols, ASPECT_COLUMNS)
    predictions = reorder_columns(predictions, aspect_cols, ASPECT_COLUMNS)

    model_name = f"MPNet {bundle_name}"
    return {
        "model_name": model_name,
        "metadata": metadata,
        "probabilities": probabilities,
        "predictions": predictions,
        "metrics": evaluate_predictions(model_name, y_test, predictions, seconds=seconds),
        "report": classification_report(
            y_test,
            predictions,
            target_names=ASPECT_COLUMNS,
            output_dict=True,
            zero_division=0,
        ),
        "prediction_frame": build_prediction_frame(model_name, probabilities, predictions),
    }


mpnet_results = {
    "MPNet unretrained": evaluate_mpnet_bundle("unretrained"),
    "MPNet retrained": evaluate_mpnet_bundle("retrained"),
}


## 5. Evaluate TF-IDF Model


In [ ]:
def evaluate_tfidf_model() -> dict:
    X_test_tfidf = sp.load_npz(TFIDF_OUTPUT_DIR / "X_test_tfidf.npz")
    classifier = joblib.load(TFIDF_CLASSIFIER_DIR / "tfidf_ovr_logreg.joblib")
    with (TFIDF_CLASSIFIER_DIR / "tfidf_metadata.json").open(encoding="utf-8") as file:
        metadata = json.load(file)

    start = time.perf_counter()
    probabilities = classifier.predict_proba(X_test_tfidf)
    seconds = time.perf_counter() - start

    predictions = apply_thresholds(probabilities, metadata["thresholds"], TFIDF_ASPECT_COLUMNS)
    probabilities = reorder_columns(probabilities, TFIDF_ASPECT_COLUMNS, ASPECT_COLUMNS)
    predictions = reorder_columns(predictions, TFIDF_ASPECT_COLUMNS, ASPECT_COLUMNS)

    model_name = "TF-IDF"
    return {
        "model_name": model_name,
        "metadata": metadata,
        "probabilities": probabilities,
        "predictions": predictions,
        "metrics": evaluate_predictions(model_name, y_test, predictions, seconds=seconds),
        "report": classification_report(
            y_test,
            predictions,
            target_names=ASPECT_COLUMNS,
            output_dict=True,
            zero_division=0,
        ),
        "prediction_frame": build_prediction_frame(model_name, probabilities, predictions),
    }


tfidf_result = evaluate_tfidf_model()
all_results = {**mpnet_results, "TF-IDF": tfidf_result}


## 6. Overall Metrics


In [ ]:
comparison_df = (
    pd.DataFrame([result["metrics"] for result in all_results.values()])
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

comparison_df.to_csv(FINAL_OUTPUT_DIR / "final_model_comparison.csv", index=False)
display(comparison_df)


In [ ]:
main_metrics = [
    "macro_f1",
    "micro_f1",
    "samples_f1",
    "exact_match_ratio",
    "hamming_loss",
]

plot_df = comparison_df.melt(
    id_vars="model",
    value_vars=main_metrics,
    var_name="metric",
    value_name="value",
)

plt.figure(figsize=(11, 5))
sns.barplot(data=plot_df, x="metric", y="value", hue="model")
plt.title("Final test-set model comparison")
plt.xlabel("Metric")
plt.ylabel("Value")
plt.xticks(rotation=20, ha="right")
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FINAL_OUTPUT_DIR / "overall_metrics_comparison.png", dpi=160)
plt.show()


In [ ]:
speed_cols = ["model", "seconds", "milliseconds_per_review"]
speed_df = comparison_df[[col for col in speed_cols if col in comparison_df.columns]].copy()
display(speed_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=speed_df, x="model", y="milliseconds_per_review")
plt.title("Inference latency per review")
plt.xlabel("Model")
plt.ylabel("Milliseconds per review")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(FINAL_OUTPUT_DIR / "latency_comparison.png", dpi=160)
plt.show()


## 7. Per-Aspect F1 Comparison


In [ ]:
aspect_f1_cols = [f"{aspect}_f1" for aspect in ASPECT_COLUMNS]
aspect_f1_df = comparison_df[["model", *aspect_f1_cols]].melt(
    id_vars="model",
    var_name="aspect",
    value_name="f1",
)
aspect_f1_df["aspect"] = aspect_f1_df["aspect"].str.replace("_f1", "", regex=False)

aspect_f1_df.to_csv(FINAL_OUTPUT_DIR / "per_aspect_f1_comparison.csv", index=False)
display(aspect_f1_df.pivot(index="aspect", columns="model", values="f1").loc[ASPECT_COLUMNS])

plt.figure(figsize=(11, 5))
sns.barplot(data=aspect_f1_df, x="aspect", y="f1", hue="model")
plt.title("Per-aspect F1 on test set")
plt.xlabel("Aspect")
plt.ylabel("F1")
plt.ylim(0, 1.05)
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FINAL_OUTPUT_DIR / "per_aspect_f1_comparison.png", dpi=160)
plt.show()


## 8. Classification Reports


In [ ]:
report_frames = []
for model_name, result in all_results.items():
    report = pd.DataFrame(result["report"]).T.reset_index().rename(columns={"index": "label"})
    report.insert(0, "model", model_name)
    report_frames.append(report)

reports_df = pd.concat(report_frames, ignore_index=True)
reports_df.to_csv(FINAL_OUTPUT_DIR / "classification_reports.csv", index=False)

display(
    reports_df[
        reports_df["label"].isin(ASPECT_COLUMNS + ["micro avg", "macro avg", "samples avg"])
    ].round(4)
)


## 9. Confusion Matrices


In [ ]:
fig, axes = plt.subplots(len(all_results), len(ASPECT_COLUMNS), figsize=(18, 10))

for row_index, (model_name, result) in enumerate(all_results.items()):
    matrices = multilabel_confusion_matrix(y_test, result["predictions"])
    for col_index, aspect in enumerate(ASPECT_COLUMNS):
        ax = axes[row_index, col_index]
        sns.heatmap(
            matrices[col_index],
            annot=True,
            fmt="d",
            cmap="Blues",
            cbar=False,
            ax=ax,
        )
        ax.set_title(f"{model_name}\n{aspect}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig(FINAL_OUTPUT_DIR / "confusion_matrices.png", dpi=160)
plt.show()


## 10. Exact-Match Overlap


In [ ]:
exact_match_df = test_df[["source_row", "review_en", "review_cleaned"]].copy()
exact_match_df["true_aspects"] = [labels_to_text(row) for row in y_test]

for model_name, result in all_results.items():
    key = model_name.lower().replace(" ", "_").replace("-", "")
    exact_match_df[f"{key}_predicted_aspects"] = [
        labels_to_text(row) for row in result["predictions"]
    ]
    exact_match_df[f"{key}_exact_match"] = (
        result["predictions"] == y_test
    ).all(axis=1)

exact_match_cols = [col for col in exact_match_df.columns if col.endswith("_exact_match")]
overlap_summary = (
    exact_match_df[exact_match_cols]
    .value_counts()
    .rename_axis(exact_match_cols)
    .reset_index(name="sample_count")
)

exact_match_df.to_csv(FINAL_OUTPUT_DIR / "sample_level_predictions.csv", index=False)
overlap_summary.to_csv(FINAL_OUTPUT_DIR / "exact_match_overlap_summary.csv", index=False)

display(overlap_summary)


## 11. Samples Correct In MPNet But Wrong In TF-IDF


In [ ]:
mpnet_key = "mpnet_retrained"
tfidf_key = "tfidf"

mpnet_correct_tfidf_wrong = exact_match_df[
    exact_match_df[f"{mpnet_key}_exact_match"]
    & ~exact_match_df[f"{tfidf_key}_exact_match"]
].copy()

tfidf_correct_mpnet_wrong = exact_match_df[
    ~exact_match_df[f"{mpnet_key}_exact_match"]
    & exact_match_df[f"{tfidf_key}_exact_match"]
].copy()

comparison_sample_cols = [
    "source_row",
    "review_en",
    "true_aspects",
    f"{mpnet_key}_predicted_aspects",
    f"{tfidf_key}_predicted_aspects",
]

mpnet_correct_tfidf_wrong[comparison_sample_cols].to_csv(
    FINAL_OUTPUT_DIR / "mpnet_retrained_correct_tfidf_wrong.csv",
    index=False,
)
tfidf_correct_mpnet_wrong[comparison_sample_cols].to_csv(
    FINAL_OUTPUT_DIR / "tfidf_correct_mpnet_retrained_wrong.csv",
    index=False,
)

print(f"MPNet retrained correct, TF-IDF wrong: {len(mpnet_correct_tfidf_wrong)}")
display(mpnet_correct_tfidf_wrong[comparison_sample_cols].head(20))


In [ ]:
print(f"TF-IDF correct, MPNet retrained wrong: {len(tfidf_correct_mpnet_wrong)}")
display(tfidf_correct_mpnet_wrong[comparison_sample_cols].head(20))


In [ ]:
counts = pd.DataFrame(
    {
        "case": [
            "MPNet retrained correct, TF-IDF wrong",
            "TF-IDF correct, MPNet retrained wrong",
        ],
        "sample_count": [
            len(mpnet_correct_tfidf_wrong),
            len(tfidf_correct_mpnet_wrong),
        ],
    }
)

plt.figure(figsize=(8, 4))
sns.barplot(data=counts, x="case", y="sample_count")
plt.title("Exact-match disagreements between retrained MPNet and TF-IDF")
plt.xlabel("")
plt.ylabel("Samples")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(FINAL_OUTPUT_DIR / "mpnet_tfidf_disagreement_counts.png", dpi=160)
plt.show()

display(counts)


## 12. Aspect-Level Error Differences


In [ ]:
error_rows = []
for model_name, result in all_results.items():
    predictions = result["predictions"]
    for index, aspect in enumerate(ASPECT_COLUMNS):
        y_true_col = y_test[:, index]
        y_pred_col = predictions[:, index]
        false_positive = int(((y_true_col == 0) & (y_pred_col == 1)).sum())
        false_negative = int(((y_true_col == 1) & (y_pred_col == 0)).sum())
        true_positive = int(((y_true_col == 1) & (y_pred_col == 1)).sum())
        true_negative = int(((y_true_col == 0) & (y_pred_col == 0)).sum())
        error_rows.append(
            {
                "model": model_name,
                "aspect": aspect,
                "true_positive": true_positive,
                "false_positive": false_positive,
                "false_negative": false_negative,
                "true_negative": true_negative,
            }
        )

error_df = pd.DataFrame(error_rows)
error_df.to_csv(FINAL_OUTPUT_DIR / "aspect_error_counts.csv", index=False)
display(error_df)


In [ ]:
error_plot_df = error_df.melt(
    id_vars=["model", "aspect"],
    value_vars=["false_positive", "false_negative"],
    var_name="error_type",
    value_name="count",
)

plt.figure(figsize=(12, 5))
sns.barplot(data=error_plot_df, x="aspect", y="count", hue="model")
plt.title("False positives + false negatives by aspect")
plt.xlabel("Aspect")
plt.ylabel("Error count")
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FINAL_OUTPUT_DIR / "aspect_total_error_comparison.png", dpi=160)
plt.show()


## 13. Best And Worst Cases

Best cases are exact-match samples where the model assigns high average probability to the true aspects.

Worst cases are non-exact-match samples ranked by:

1. more label errors first
2. higher wrong-confidence first
3. lower true-aspect confidence first

These tables are useful for explaining where each model is reliable and where it fails.


In [ ]:
def aspect_list_from_binary(row: np.ndarray) -> list[str]:
    return [aspect for aspect, value in zip(ASPECT_COLUMNS, row.astype(int)) if value == 1]


def build_case_analysis_frame(model_name: str, result: dict) -> pd.DataFrame:
    probabilities = result["probabilities"]
    predictions = result["predictions"]

    rows = []
    for row_index in range(len(test_df)):
        true_row = y_test[row_index].astype(int)
        pred_row = predictions[row_index].astype(int)
        prob_row = probabilities[row_index]

        false_positive_aspects = [
            aspect
            for aspect_index, aspect in enumerate(ASPECT_COLUMNS)
            if true_row[aspect_index] == 0 and pred_row[aspect_index] == 1
        ]
        false_negative_aspects = [
            aspect
            for aspect_index, aspect in enumerate(ASPECT_COLUMNS)
            if true_row[aspect_index] == 1 and pred_row[aspect_index] == 0
        ]
        true_aspect_mask = true_row.astype(bool)
        predicted_aspect_mask = pred_row.astype(bool)
        wrong_predicted_mask = (true_row == 0) & (pred_row == 1)

        mean_true_probability = (
            float(prob_row[true_aspect_mask].mean())
            if true_aspect_mask.any()
            else np.nan
        )
        min_true_probability = (
            float(prob_row[true_aspect_mask].min())
            if true_aspect_mask.any()
            else np.nan
        )
        mean_predicted_probability = (
            float(prob_row[predicted_aspect_mask].mean())
            if predicted_aspect_mask.any()
            else np.nan
        )
        max_wrong_probability = (
            float(prob_row[wrong_predicted_mask].max())
            if wrong_predicted_mask.any()
            else 0.0
        )

        rows.append(
            {
                "model": model_name,
                "source_row": int(test_df.loc[row_index, "source_row"]),
                "review_en": test_df.loc[row_index, "review_en"],
                "true_aspects": labels_to_text(true_row),
                "predicted_aspects": labels_to_text(pred_row),
                "exact_match": bool((true_row == pred_row).all()),
                "label_error_count": int(np.abs(true_row - pred_row).sum()),
                "false_positive_aspects": ", ".join(false_positive_aspects) or "None",
                "false_negative_aspects": ", ".join(false_negative_aspects) or "None",
                "mean_true_probability": mean_true_probability,
                "min_true_probability": min_true_probability,
                "mean_predicted_probability": mean_predicted_probability,
                "max_wrong_probability": max_wrong_probability,
            }
        )

    return pd.DataFrame(rows)


case_frames = [
    build_case_analysis_frame(model_name, result)
    for model_name, result in all_results.items()
]
case_analysis_df = pd.concat(case_frames, ignore_index=True)
case_analysis_df.to_csv(FINAL_OUTPUT_DIR / "case_analysis_all_models.csv", index=False)

display(case_analysis_df.head())


In [ ]:
best_case_frames = []
worst_case_frames = []

for model_name in all_results:
    model_cases = case_analysis_df[case_analysis_df["model"] == model_name].copy()

    best_cases = (
        model_cases[model_cases["exact_match"]]
        .sort_values(
            ["mean_true_probability", "min_true_probability"],
            ascending=[False, False],
        )
        .head(15)
    )
    worst_cases = (
        model_cases[~model_cases["exact_match"]]
        .sort_values(
            [
                "label_error_count",
                "max_wrong_probability",
                "mean_true_probability",
            ],
            ascending=[False, False, True],
        )
        .head(15)
    )

    safe_name = model_name.lower().replace(" ", "_").replace("-", "")
    best_cases.to_csv(FINAL_OUTPUT_DIR / f"{safe_name}_best_cases.csv", index=False)
    worst_cases.to_csv(FINAL_OUTPUT_DIR / f"{safe_name}_worst_cases.csv", index=False)

    best_case_frames.append(best_cases)
    worst_case_frames.append(worst_cases)

all_best_cases = pd.concat(best_case_frames, ignore_index=True)
all_worst_cases = pd.concat(worst_case_frames, ignore_index=True)

all_best_cases.to_csv(FINAL_OUTPUT_DIR / "best_cases_all_models.csv", index=False)
all_worst_cases.to_csv(FINAL_OUTPUT_DIR / "worst_cases_all_models.csv", index=False)

print("Saved best/worst case files to:", FINAL_OUTPUT_DIR)


In [ ]:
case_summary = (
    case_analysis_df
    .groupby("model")
    .agg(
        exact_matches=("exact_match", "sum"),
        total_samples=("exact_match", "size"),
        mean_label_errors=("label_error_count", "mean"),
        max_label_errors=("label_error_count", "max"),
        mean_true_probability=("mean_true_probability", "mean"),
        mean_predicted_probability=("mean_predicted_probability", "mean"),
        mean_max_wrong_probability=("max_wrong_probability", "mean"),
    )
    .reset_index()
)
case_summary["exact_match_ratio"] = (
    case_summary["exact_matches"] / case_summary["total_samples"]
)
case_summary.to_csv(FINAL_OUTPUT_DIR / "case_analysis_summary.csv", index=False)
display(case_summary.round(4))


In [ ]:
plt.figure(figsize=(9, 4))
sns.barplot(data=case_summary, x="model", y="mean_label_errors")
plt.title("Mean label errors per sample")
plt.xlabel("Model")
plt.ylabel("Mean label errors")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(FINAL_OUTPUT_DIR / "mean_label_errors_by_model.png", dpi=160)
plt.show()


### Best cases by model


In [ ]:
best_display_cols = [
    "model",
    "source_row",
    "review_en",
    "true_aspects",
    "predicted_aspects",
    "mean_true_probability",
    "min_true_probability",
]

display(
    all_best_cases[best_display_cols]
    .sort_values(["model", "mean_true_probability"], ascending=[True, False])
    .groupby("model")
    .head(8)
    .round(4)
)


### Worst cases by model


In [ ]:
worst_display_cols = [
    "model",
    "source_row",
    "review_en",
    "true_aspects",
    "predicted_aspects",
    "label_error_count",
    "false_positive_aspects",
    "false_negative_aspects",
    "max_wrong_probability",
    "mean_true_probability",
]

display(
    all_worst_cases[worst_display_cols]
    .sort_values(
        ["model", "label_error_count", "max_wrong_probability"],
        ascending=[True, False, False],
    )
    .groupby("model")
    .head(8)
    .round(4)
)


### How To Read These Cases

- High-confidence best cases show the patterns each model handles reliably.
- Worst cases with many false positives indicate over-prediction.
- Worst cases with many false negatives indicate missed aspects.
- For this project, inspect `miscellaneous` mistakes first because it is the rarest and weakest class.
